# Project 1 v3: Production VaR Engine — Full Rebuild

**Improvements over v1/v2:**
- ✅ Copula-based Monte Carlo (Gaussian vs t-Copula with formal selection test)
- ✅ Per-asset GARCH(1,1) + DCC (Dynamic Conditional Correlation) for time-varying covariance
- ✅ Covariance matrix positive-definiteness check + Ledoit-Wolf shrinkage fallback
- ✅ Stress window fixed to exactly 250 trading days
- ✅ Dynamic bid-ask spread via CBOE VIX (liquidity proxy) with static fallback
- ✅ Delta-Gamma approximation for USO and GLD (commodity non-linearity)
- ✅ Corrected Christoffersen test (numpy vectorized)
- ✅ Proper GARCH warm-up (rolling 30-day init)
- ✅ Global random seed management
- ✅ Shapiro-Wilk replaced by Ljung-Box on z² for ARCH residual diagnostics
- ✅ Expected Shortfall (ES) alongside VaR throughout

---
### Setup
```
pip install yfinance pandas numpy scipy statsmodels arch matplotlib
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import norm, t as studentt, chi2
from scipy.optimize import minimize
from scipy.linalg import cholesky as sp_cholesky
import os

import yfinance as yf

# statsmodels for Ljung-Box
try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('WARNING: statsmodels not installed — Ljung-Box test will be skipped.')
    print('Install with: pip install statsmodels')

# arch for per-asset GARCH (optional, falls back to hand-coded)
try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    HAS_ARCH = False
    print('WARNING: arch package not installed — using hand-coded GARCH.')
    print('Install with: pip install arch')

# ── Global seed ──────────────────────────────────────────────────────────────
GLOBAL_SEED = 42
RNG = np.random.default_rng(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#2d2d44',
    'axes.edgecolor':   '#3a3a55',
    'text.color':       '#e8e8e8',
    'xtick.color':      '#a0a0b8',
    'ytick.color':      '#a0a0b8',
    'grid.color':       '#3a3a55',
    'grid.linewidth':   0.5,
})

os.makedirs('output', exist_ok=True)
print('Libraries loaded. Global seed:', GLOBAL_SEED)

## 1. Configuration

In [ ]:
PORTFOLIO = {
    'SPY': {'weight': 0.20, 'asset_class': 'Equity',    'name': 'S&P 500 ETF'},
    'QQQ': {'weight': 0.10, 'asset_class': 'Equity',    'name': 'Nasdaq 100 ETF'},
    'EEM': {'weight': 0.05, 'asset_class': 'EM Equity', 'name': 'EM Equity ETF'},
    'TLT': {'weight': 0.20, 'asset_class': 'Rates',     'name': '20yr Treasury ETF'},
    'LQD': {'weight': 0.10, 'asset_class': 'Credit',    'name': 'IG Corporate ETF'},
    'HYG': {'weight': 0.05, 'asset_class': 'Credit',    'name': 'HY Corporate ETF'},
    'GLD': {'weight': 0.15, 'asset_class': 'Commodity', 'name': 'Gold ETF',
            'nonlinear': True,  # Use delta-gamma approximation
            'gamma_proxy': 0.02},  # Proxy gamma (convexity coeff, calibrate to options)
    'USO': {'weight': 0.10, 'asset_class': 'Commodity', 'name': 'Oil ETF',
            'nonlinear': True,
            'gamma_proxy': 0.04},  # Oil has higher convexity/skewness exposure
    'FXE': {'weight': 0.05, 'asset_class': 'FX',        'name': 'EUR/USD ETF'},
}

PORTFOLIO_VALUE  = 100_000_000
VAR_CONFIDENCE   = 0.99
HS_WINDOW        = 250          # Exactly 250 trading days
MC_SIMULATIONS   = 50_000
START_DATE       = '2005-01-01'
END_DATE         = '2024-12-31'

# Static bid-ask spreads (fallback if dynamic data unavailable)
# Units: fraction of price (e.g. 0.005 = 50bps)
BID_ASK_SPREADS_STATIC = {
    'Equity':    0.005,
    'EM Equity': 0.015,
    'Rates':     0.003,
    'Credit':    0.020,
    'Commodity': 0.010,
    'FX':        0.002,
}
LIQUIDATION_DAYS = {
    'Equity': 1, 'EM Equity': 3, 'Rates': 1,
    'Credit': 5, 'Commodity': 2, 'FX': 1,
}

print('Config loaded. Portfolio weights:')
for t, v in PORTFOLIO.items():
    nl = ' [delta-gamma]' if v.get('nonlinear') else ''
    print(f"  {t:5s}  {v['weight']*100:5.1f}%  {v['asset_class']:12s}  {v['name']}{nl}")

## 2. Data Acquisition

In [ ]:
tickers = list(PORTFOLIO.keys())
print(f'Fetching price data for: {tickers}')

prices = yf.download(
    tickers, start=START_DATE, end=END_DATE,
    auto_adjust=True, progress=True
)['Close']

# ── Data quality checks ───────────────────────────────────────────────────────
# 1. Forward-fill gaps up to 3 days (handles holidays, early closes)
prices = prices.ffill(limit=3)

# 2. Report missing data before dropping
missing = prices.isna().sum()
if missing.any():
    print('\nMissing values after forward-fill:')
    print(missing[missing > 0])

# 3. Drop rows where ANY ticker is still NaN (need complete cross-section)
prices = prices.dropna(how='any')

prices.to_csv('output/prices_raw.csv')
print(f'\nShape: {prices.shape}  ({prices.shape[0]} trading days)')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

# Compute daily log returns
returns = np.log(prices / prices.shift(1)).dropna()

# Return summary
summary = returns.describe().T
summary['skew']     = returns.skew()
summary['kurtosis'] = returns.kurtosis()  # excess kurtosis
print('\nReturn statistics (kurtosis = excess):')
print(summary[['mean','std','skew','kurtosis','min','max']].round(5))

## 3. Dynamic Bid-Ask Spread via VIX

Bid-ask spreads widen significantly in stress: empirically, credit spreads can expand 5-10x in a crisis.
We use VIX as a liquidity proxy: `spread_dynamic = spread_static × (1 + VIX_scaling_factor)`.

**VIX scaling:** At VIX=20 (calm), multiplier = 1.0. At VIX=80 (GFC peak), multiplier ≈ 4.0.
This is a parsimonious proxy — production systems would use TRACE data for credit, TAQ for equity.

In [ ]:
def fetch_vix(start_date, end_date):
    """Fetch VIX index. Returns None if unavailable."""
    try:
        vix = yf.download('^VIX', start=start_date, end=end_date,
                          auto_adjust=True, progress=False)['Close']
        vix = vix.ffill(limit=3).dropna()
        if len(vix) < 100:
            return None
        print(f'VIX data loaded: {len(vix)} observations')
        print(f'  Range: {vix.min():.1f} – {vix.max():.1f}')
        return vix
    except Exception as e:
        print(f'VIX fetch failed: {e}. Using static spreads.')
        return None

vix_series = fetch_vix(START_DATE, END_DATE)
VIX_CALM_LEVEL = 20.0   # Baseline VIX assumed in static spreads

def get_dynamic_spread(asset_class, date, vix_series=vix_series):
    """
    Returns the estimated bid-ask spread for a given asset class on a given date.
    If VIX data is available, scales the static spread by VIX regime.
    Credit spreads are more sensitive to vol than equity; FX is least sensitive.
    """
    base = BID_ASK_SPREADS_STATIC[asset_class]
    if vix_series is None:
        return base
    try:
        # Use nearest available VIX observation
        vix_val = vix_series.asof(date) if hasattr(vix_series, 'asof') else vix_series.get(date, VIX_CALM_LEVEL)
        if pd.isna(vix_val):
            return base
    except Exception:
        return base

    # Sensitivity multipliers by asset class (calibrated to crisis observations)
    vix_sensitivity = {
        'Equity':    0.05,   # moderate: price impact mainly from volume
        'EM Equity': 0.08,
        'Rates':     0.02,   # liquid: tight even in crisis
        'Credit':    0.15,   # most sensitive: illiquidity spiral in stress
        'Commodity': 0.06,
        'FX':        0.02,
    }
    sens = vix_sensitivity.get(asset_class, 0.05)
    multiplier = 1.0 + sens * max(0, vix_val - VIX_CALM_LEVEL)
    return base * multiplier

# Show current spreads (latest date)
latest_date = returns.index[-1]
print('\nBid-ask spreads (latest date):')
for t, info in PORTFOLIO.items():
    ac = info['asset_class']
    static  = BID_ASK_SPREADS_STATIC[ac]
    dynamic = get_dynamic_spread(ac, latest_date)
    print(f'  {t:5s}  {ac:12s}  static={static*100:.3f}%  dynamic={dynamic*100:.3f}%')

## 4. Covariance Matrix: Positive-Definiteness Check + Shrinkage

A sample covariance matrix can be near-singular when:
- Number of assets approaches number of observations
- Multicollinearity exists (e.g. SPY/QQQ are highly correlated)

We test eigenvalues and apply **Ledoit-Wolf shrinkage** if the matrix is ill-conditioned.

In [ ]:
def check_and_fix_covariance(R, name='sample', shrink_threshold=1e-6):
    """
    R: (T x N) array of returns
    Returns: (N x N) positive-definite covariance matrix + diagnostics dict
    """
    n_obs, n_assets = R.shape
    cov_raw = np.cov(R.T)

    eigenvalues = np.linalg.eigvalsh(cov_raw)
    min_eig = eigenvalues.min()
    condition_number = eigenvalues.max() / max(abs(min_eig), 1e-12)

    diag = {
        'n_obs':            n_obs,
        'n_assets':         n_assets,
        'min_eigenvalue':   min_eig,
        'condition_number': condition_number,
        'is_pd':            bool(min_eig > 0),
        'method_used':      'sample',
    }

    print(f'\nCovariance matrix [{name}]:')
    print(f'  Observations: {n_obs}, Assets: {n_assets}')
    print(f'  Min eigenvalue: {min_eig:.2e}  (must be > 0 for PD)')
    print(f'  Condition number: {condition_number:.1f}  (> 1e6 is ill-conditioned)')
    print(f'  Positive definite: {diag["is_pd"]}')

    if not diag['is_pd'] or condition_number > 1e6 or min_eig < shrink_threshold:
        print('  → Applying Ledoit-Wolf shrinkage...')
        cov_fixed = ledoit_wolf_shrinkage(R)
        # Verify after shrinkage
        eig_fixed = np.linalg.eigvalsh(cov_fixed)
        diag['method_used']      = 'ledoit_wolf'
        diag['min_eig_after']    = eig_fixed.min()
        diag['is_pd_after']      = bool(eig_fixed.min() > 0)
        print(f'  → Post-shrinkage min eigenvalue: {eig_fixed.min():.2e}')
        return cov_fixed, diag
    else:
        print('  ✓ Matrix is well-conditioned, no adjustment needed.')
        return cov_raw, diag


def ledoit_wolf_shrinkage(R):
    """
    Analytical Ledoit-Wolf (2004) shrinkage toward scaled identity.
    Shrinks sample covariance toward (mu * I) where mu = trace(S)/p.
    Formula: Sigma* = (1 - alpha)*S + alpha*mu*I
    Alpha is chosen to minimise asymptotic MSE.
    """
    T, p = R.shape
    S = np.cov(R.T)
    mu = np.trace(S) / p

    # Estimate optimal shrinkage intensity (Oracle approximating shrinkage)
    delta2 = 0.0
    for i in range(T):
        xi = R[i:i+1, :].T
        delta2 += np.sum((xi @ xi.T - S)**2)
    delta2 /= T**2

    # Ledoit-Wolf optimal alpha
    gamma = np.sum((S - mu * np.eye(p))**2)
    alpha = min(delta2 / gamma, 1.0) if gamma > 0 else 0.1

    cov_shrunk = (1 - alpha) * S + alpha * mu * np.eye(p)
    print(f'  Ledoit-Wolf shrinkage intensity alpha = {alpha:.4f}')
    return cov_shrunk


# ── Run check ────────────────────────────────────────────────────────────────
tickers_in = [t for t in PORTFOLIO if t in returns.columns]
weights = pd.Series({t: PORTFOLIO[t]['weight'] for t in tickers_in})
weights /= weights.sum()
w_arr = weights.values

R_full = returns[tickers_in].values
cov_matrix, cov_diag = check_and_fix_covariance(R_full, name='full sample')

# Per-asset marginal statistics for copula fitting
mu_vec = returns[tickers_in].mean().values

## 5. Per-Asset GARCH(1,1) and DCC — Time-Varying Correlation

**Why per-asset GARCH + DCC instead of a single covariance matrix?**

A static covariance matrix ignores:
1. **Volatility clustering** — each asset has its own conditional heteroskedasticity
2. **Time-varying correlation** — correlations spike in crises ("correlation goes to 1")

**DCC (Engle 2002)** decomposes: $H_t = D_t R_t D_t$  
where $D_t = \text{diag}(\sigma_{1,t}, ..., \sigma_{n,t})$ (GARCH conditional vols)  
and $R_t$ is the dynamic correlation matrix.

The correlation dynamics follow:  
$Q_t = (1-a-b)\bar{Q} + a\,\tilde{z}_{t-1}\tilde{z}_{t-1}' + b\,Q_{t-1}$  
$R_t = \text{diag}(Q_t)^{-1/2}\, Q_t\, \text{diag}(Q_t)^{-1/2}$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5a. Hand-coded GARCH(1,1) MLE with proper warm-up
# ─────────────────────────────────────────────────────────────────────────────

class GARCH11:
    """
    GARCH(1,1) with hand-coded MLE.
    Improvements over v1/v2:
      - Warm-up: sigma2[0] uses rolling 30-day variance (not full-sample var)
      - Multiple random starting points to avoid local optima
      - Returns both in-sample sigma2 and standardised residuals z = r/sigma
    """
    def __init__(self, warmup=30):
        self.warmup = warmup

    def _sigma2(self, r, omega, alpha, beta):
        n = len(r)
        s2 = np.empty(n)
        # Warm-up: use rolling window variance for s2[0]
        s2[0] = np.var(r[:self.warmup]) if len(r) >= self.warmup else np.var(r)
        for t in range(1, n):
            s2[t] = omega + alpha * r[t-1]**2 + beta * s2[t-1]
            # Safety clamp to avoid numerical underflow
            if s2[t] <= 0:
                s2[t] = 1e-12
        return s2

    def _nll(self, params, r):
        omega, alpha, beta = params
        if omega <= 0 or alpha <= 0 or beta <= 0 or alpha + beta >= 1:
            return 1e10
        s2 = self._sigma2(r, omega, alpha, beta)
        if np.any(s2 <= 0) or np.any(~np.isfinite(s2)):
            return 1e10
        ll = 0.5 * np.sum(np.log(s2) + r**2 / s2)
        return ll if np.isfinite(ll) else 1e10

    def fit(self, r):
        r = np.asarray(r, float)
        v0 = np.var(r)
        # Multiple starting points
        starts = [
            [v0 * 0.05, 0.05, 0.90],
            [v0 * 0.10, 0.10, 0.85],
            [v0 * 0.10, 0.15, 0.80],
            [v0 * 0.20, 0.08, 0.88],
            [v0 * 0.02, 0.03, 0.95],
        ]
        best_nll, best_p = np.inf, None
        for p0 in starts:
            try:
                res = minimize(
                    self._nll, p0, args=(r,), method='L-BFGS-B',
                    bounds=[(1e-8, None), (1e-6, 0.50), (1e-6, 0.999)],
                    options={'maxiter': 3000, 'ftol': 1e-14, 'gtol': 1e-8},
                )
                if res.fun < best_nll and res.success:
                    best_nll, best_p = res.fun, res.x
            except Exception:
                continue
        if best_p is None:
            # Fallback: use robust starting point regardless of convergence
            res = minimize(self._nll, [v0*0.1, 0.08, 0.88], args=(r,), method='L-BFGS-B',
                           bounds=[(1e-8, None),(1e-6,0.50),(1e-6,0.999)])
            best_p = res.x

        self.omega_, self.alpha_, self.beta_ = best_p
        self.sigma2_ = self._sigma2(r, *best_p)
        self.sigma_   = np.sqrt(self.sigma2_)
        self.resid_   = r / self.sigma_   # standardised residuals z_t
        self.r_       = r
        self.nll_     = best_nll
        return self

    @property
    def persistence(self): return self.alpha_ + self.beta_

    @property
    def long_run_ann_vol(self):
        return np.sqrt(self.omega_ / (1 - self.persistence) * 252) * 100

    def forecast_sigma(self, h=1):
        """h-step-ahead variance forecast via mean reversion."""
        lr_var = self.omega_ / (1 - self.persistence)
        s2_1   = self.omega_ + self.alpha_ * self.r_[-1]**2 + self.beta_ * self.sigma2_[-1]
        if h == 1:
            return np.sqrt(s2_1)
        # h-step via mean reversion: E[sigma^2_{t+h}] = LR + (s2_1 - LR)*persistence^(h-1)
        s2_h = lr_var + (s2_1 - lr_var) * self.persistence ** (h - 1)
        return np.sqrt(max(s2_h, 1e-12))


# ─────────────────────────────────────────────────────────────────────────────
# 5b. Fit per-asset GARCH(1,1)
# ─────────────────────────────────────────────────────────────────────────────
print('Fitting per-asset GARCH(1,1)...')
garch_models = {}

for ticker in tickers_in:
    r_i = returns[ticker].values
    if HAS_ARCH:
        # Use arch package if available (faster, more robust)
        am = arch_model(r_i * 100, vol='Garch', p=1, q=1, mean='Zero', dist='Normal')
        res = am.fit(disp='off', show_warning=False)
        # Wrap into our interface
        class ArchWrapper:
            def __init__(self, res, r_orig):
                self.omega_ = res.params['omega'] / 1e4
                self.alpha_ = res.params.get('alpha[1]', res.params.get('alpha', 0.08))
                self.beta_  = res.params.get('beta[1]',  res.params.get('beta',  0.88))
                self.sigma2_ = (res.conditional_volatility / 100)**2
                self.sigma_  = res.conditional_volatility / 100
                self.resid_  = r_orig / self.sigma_
                self.r_      = r_orig
                self.nll_    = -res.loglikelihood
            @property
            def persistence(self): return self.alpha_ + self.beta_
            @property
            def long_run_ann_vol(self):
                return np.sqrt(self.omega_ / max(1 - self.persistence, 1e-6) * 252) * 100
            def forecast_sigma(self, h=1):
                lr_var = self.omega_ / max(1 - self.persistence, 1e-6)
                s2_1   = self.omega_ + self.alpha_ * self.r_[-1]**2 + self.beta_ * self.sigma2_[-1]
                if h == 1: return np.sqrt(s2_1)
                s2_h = lr_var + (s2_1 - lr_var) * self.persistence ** (h-1)
                return np.sqrt(max(s2_h, 1e-12))
        garch_models[ticker] = ArchWrapper(res, r_i)
    else:
        g = GARCH11().fit(r_i)
        garch_models[ticker] = g

print('\nGARCH results per asset:')
print(f'{"Ticker":6s}  {"omega":>10s}  {"alpha":>6s}  {"beta":>6s}  {"alpha+beta":>10s}  {"LR Ann Vol":>10s}')
for ticker, g in garch_models.items():
    print(f'{ticker:6s}  {g.omega_:10.6f}  {g.alpha_:6.4f}  {g.beta_:6.4f}  '
          f'{g.persistence:10.4f}  {g.long_run_ann_vol:9.1f}%')
    if g.persistence >= 1.0:
        print(f'  WARNING: {ticker} has persistence >= 1 — non-stationary!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5c. GARCH residual diagnostics — Ljung-Box on z² (tests for remaining ARCH)
# ─────────────────────────────────────────────────────────────────────────────
# FIX (v1/v2): Previously ran Shapiro-Wilk (normality) instead of Ljung-Box (ARCH effects)
# The correct diagnostic is to test z_t^2 for autocorrelation.
# H0: No remaining ARCH effects. Rejection means GARCH didn't fully capture vol clustering.

print('GARCH residual diagnostics (Ljung-Box on z²):')
print(f'{"Ticker":6s}  {"LB stat":>8s}  {"p-value":>8s}  {"ARCH remaining?":>16s}')

for ticker, g in garch_models.items():
    z2 = g.resid_**2
    if HAS_STATSMODELS:
        lb = acorr_ljungbox(z2, lags=10, return_df=True)
        lb_stat = lb['lb_stat'].iloc[-1]
        lb_pval = lb['lb_pvalue'].iloc[-1]
    else:
        # Manual Ljung-Box if statsmodels unavailable
        n = len(z2)
        z2_demeaned = z2 - z2.mean()
        lb_stat = 0.0
        for lag in range(1, 11):
            rho = np.corrcoef(z2_demeaned[:-lag], z2_demeaned[lag:])[0,1]
            lb_stat += rho**2 / (n - lag)
        lb_stat *= n * (n + 2)
        lb_pval  = 1 - chi2.cdf(lb_stat, df=10)

    flag = '⚠ YES' if lb_pval < 0.05 else '✓ No'
    print(f'{ticker:6s}  {lb_stat:8.2f}  {lb_pval:8.4f}  {flag:>16s}')

print('\np < 0.05 → ARCH effects remain. Consider GARCH(2,1) or EGARCH for those assets.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5d. DCC (Dynamic Conditional Correlation) — Engle (2002)
# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Standardise returns by GARCH conditional vol → z_it = r_it / sigma_it
# Step 2: Fit DCC parameters (a, b) by quasi-MLE on the standardised residuals
# Step 3: Extract time-varying correlation matrices R_t

class DCC:
    """
    Engle (2002) DCC(1,1).
    Q_t = (1 - a - b) * Q_bar + a * z_{t-1} z_{t-1}' + b * Q_{t-1}
    R_t = diag(Q_t)^{-1/2} Q_t diag(Q_t)^{-1/2}
    """
    def _Qpath(self, Z, a, b):
        T, N = Z.shape
        Qbar = Z.T @ Z / T
        Q    = np.copy(Qbar)
        Qt   = np.empty((T, N, N))
        Qt[0] = Qbar
        for t in range(1, T):
            Q = (1 - a - b) * Qbar + a * np.outer(Z[t-1], Z[t-1]) + b * Q
            Qt[t] = Q
        return Qt, Qbar

    def _nll(self, params, Z):
        a, b = params
        if a <= 0 or b <= 0 or a + b >= 1:
            return 1e10
        Qt, _ = self._Qpath(Z, a, b)
        T, N  = Z.shape
        ll = 0.0
        for t in range(T):
            Q_t  = Qt[t]
            D_inv = 1.0 / np.sqrt(np.diag(Q_t))
            R_t  = D_inv[:, None] * Q_t * D_inv[None, :]
            np.fill_diagonal(R_t, 1.0)
            sign, logdet = np.linalg.slogdet(R_t)
            if sign <= 0:
                return 1e10
            zt   = Z[t]
            try:
                R_inv = np.linalg.inv(R_t)
            except np.linalg.LinAlgError:
                return 1e10
            ll += logdet + zt @ R_inv @ zt - zt @ zt
        return 0.5 * ll

    def fit(self, Z):
        """Z: (T x N) matrix of GARCH standardised residuals."""
        best_nll, best_p = np.inf, [0.05, 0.90]
        for p0 in [[0.05, 0.90], [0.02, 0.95], [0.10, 0.85]]:
            try:
                res = minimize(self._nll, p0, args=(Z,), method='L-BFGS-B',
                               bounds=[(1e-4, 0.5), (1e-4, 0.999)],
                               options={'maxiter': 500, 'ftol': 1e-10})
                if res.fun < best_nll:
                    best_nll, best_p = res.fun, res.x
            except Exception:
                continue
        self.a_, self.b_ = best_p
        self.Qt_, self.Qbar_ = self._Qpath(Z, self.a_, self.b_)
        T, N = Z.shape
        self.Rt_ = np.empty((T, N, N))
        for t in range(T):
            Q_t   = self.Qt_[t]
            D_inv = 1.0 / np.sqrt(np.diag(Q_t))
            R_t   = D_inv[:, None] * Q_t * D_inv[None, :]
            np.fill_diagonal(R_t, 1.0)
            self.Rt_[t] = R_t
        self.Z_ = Z
        return self

    def forecast_H(self, garch_sigmas_next):
        """One-step-ahead conditional covariance H_{T+1}."""
        a, b     = self.a_, self.b_
        z_last   = self.Z_[-1]
        Q_last   = self.Qt_[-1]
        Q_next   = (1 - a - b) * self.Qbar_ + a * np.outer(z_last, z_last) + b * Q_last
        D_inv    = 1.0 / np.sqrt(np.diag(Q_next))
        R_next   = D_inv[:, None] * Q_next * D_inv[None, :]
        np.fill_diagonal(R_next, 1.0)
        D_sigma  = np.diag(garch_sigmas_next)
        H_next   = D_sigma @ R_next @ D_sigma
        return H_next, R_next


# ── Collect GARCH standardised residuals ─────────────────────────────────────
Z_matrix = np.column_stack([garch_models[t].resid_ for t in tickers_in])

print(f'Fitting DCC on {Z_matrix.shape[0]} obs × {Z_matrix.shape[1]} assets...')
print('(This may take 1-2 minutes)')
dcc = DCC().fit(Z_matrix)

print(f'\nDCC parameters:')
print(f'  a = {dcc.a_:.4f}  (shock transmission to correlation)')
print(f'  b = {dcc.b_:.4f}  (correlation persistence)')
print(f'  a + b = {dcc.a_+dcc.b_:.4f}  (stationarity: must be < 1)')

# One-step-ahead forecast
garch_sigmas_next = np.array([garch_models[t].forecast_sigma(h=1) for t in tickers_in])
H_next, R_next = dcc.forecast_H(garch_sigmas_next)

print(f'\nForward-looking DCC correlation matrix (latest):')
corr_df = pd.DataFrame(R_next, index=tickers_in, columns=tickers_in)
print(corr_df.round(3))

In [ ]:
# Visualise: correlation evolution over time for key pairs
key_pairs = [('SPY', 'TLT'), ('SPY', 'HYG'), ('GLD', 'SPY'), ('USO', 'SPY')]
pair_indices = [(tickers_in.index(a), tickers_in.index(b)) for a, b in key_pairs
                if a in tickers_in and b in tickers_in]

fig, ax = plt.subplots(figsize=(15, 4))
colors = ['#4a90d9', '#e8734a', '#4ac9a0', '#f5c842']
for (i, j), (a, b), col in zip(pair_indices, key_pairs, colors):
    rho_series = dcc.Rt_[:, i, j]
    ax.plot(returns.index, rho_series, lw=0.8, color=col, label=f'{a}/{b}', alpha=0.9)

ax.axhline(0, color='white', lw=0.5, linestyle=':')
ax.set_ylabel('Dynamic Correlation')
ax.set_title('DCC Time-Varying Correlations — Key Asset Pairs', pad=8)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/dcc_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Copula Selection: Gaussian vs t-Copula

**Why Copula?**  
Copulas separate the marginal distributions from the dependence structure (Sklar's theorem).  
This allows: each asset has its own tail shape (fitted t-distribution) + the joint dependence is modelled by a copula.

**Gaussian Copula:** Tail independence — joint tail events are asymptotically independent.  
**t-Copula:** Tail dependence — extreme events tend to co-occur across assets. More realistic for crises.

**Selection test:** We use two formal approaches:
1. **Tail dependence coefficient** — empirical λ_U vs copula-implied λ_U
2. **AIC/BIC comparison** — fit both copulas by MLE, select by information criterion
3. **Kolmogorov-Smirnov test** — compare simulated vs empirical joint tail distributions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6a. Transform marginals to uniform via PIT (Probability Integral Transform)
# ─────────────────────────────────────────────────────────────────────────────
# We use GARCH residuals z_t (already vol-normalised) and fit per-asset t distribution

print('Step 1: Fit per-asset marginal distributions (Student-t)...')
marginal_params = {}  # {ticker: (df, loc, scale)}
U_matrix = np.empty_like(Z_matrix)   # uniform pseudo-observations after PIT

for j, ticker in enumerate(tickers_in):
    z = Z_matrix[:, j]
    # Fit t distribution to GARCH residuals
    df_fit, loc_fit, scale_fit = studentt.fit(z, floc=0)  # fix loc=0 for residuals
    marginal_params[ticker] = (df_fit, loc_fit, scale_fit)
    # PIT: transform to uniform [0,1]
    U_matrix[:, j] = studentt.cdf(z, df=df_fit, loc=loc_fit, scale=scale_fit)
    # Clip to avoid exact 0 or 1 (numerical issues in inverse CDF)
    U_matrix[:, j] = np.clip(U_matrix[:, j], 1e-6, 1 - 1e-6)

print(f'{"Ticker":6s}  {"df":>8s}  {"loc":>8s}  {"scale":>8s}')
for t, (df, loc, sc) in marginal_params.items():
    excess_kurt = 6 / (df - 4) if df > 4 else float('inf')
    print(f'{t:6s}  {df:8.2f}  {loc:8.4f}  {sc:8.4f}   (excess kurtosis ≈ {excess_kurt:.2f})')

# Transform uniform to standard normal for copula estimation
W_matrix = norm.ppf(U_matrix)  # Gaussian copula latent variables

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6b. Empirical tail dependence
# ─────────────────────────────────────────────────────────────────────────────
def empirical_tail_dependence(U, threshold=0.05):
    """
    Estimate lower tail dependence coefficient λ_L for all pairs.
    λ_L = P(U_i < u, U_j < u) / u  as u → 0
    Uses non-parametric rank-based estimate at threshold u.
    """
    T, N = U.shape
    lambda_L = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            joint_low = np.mean((U[:, i] <= threshold) & (U[:, j] <= threshold))
            lam = joint_low / threshold
            lambda_L[i, j] = lambda_L[j, i] = lam
    return lambda_L

lambda_L_emp = empirical_tail_dependence(U_matrix, threshold=0.05)

# For a Gaussian copula, tail dependence is always 0 (asymptotically)
# For a t-copula with df v and correlation rho:
# lambda_L = 2 * t_{v+1}( -sqrt((v+1)*(1-rho)/(1+rho)) )
# If empirical lambda_L > 0 significantly → t-copula preferred

# Corr matrix from W (Gaussian copula correlation)
R_gaussian = np.corrcoef(W_matrix.T)

# Average pairwise empirical tail dependence (upper triangle only)
upper_idx = np.triu_indices(len(tickers_in), k=1)
avg_lambda = lambda_L_emp[upper_idx].mean()
max_lambda = lambda_L_emp[upper_idx].max()

print(f'Empirical lower tail dependence (threshold=5%):')
lam_df = pd.DataFrame(lambda_L_emp.round(3), index=tickers_in, columns=tickers_in)
print(lam_df)
print(f'\nAverage λ_L = {avg_lambda:.3f}   Max λ_L = {max_lambda:.3f}')
print('Interpretation: λ_L > 0.05 across many pairs → t-copula preferred over Gaussian')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6c. Fit Gaussian and t-Copula; compare via AIC/BIC
# ─────────────────────────────────────────────────────────────────────────────

def gaussian_copula_loglik(R, W):
    """
    Log-likelihood of Gaussian copula.
    l = -T/2 * log|R| - 1/2 * sum_t (w_t' R^{-1} w_t - w_t' w_t)
    """
    T, N = W.shape
    sign, logdet = np.linalg.slogdet(R)
    if sign <= 0:
        return -np.inf
    try:
        R_inv = np.linalg.inv(R)
    except np.linalg.LinAlgError:
        return -np.inf
    quad = np.einsum('ti,ij,tj->t', W, R_inv, W)   # w_t' R^{-1} w_t for all t
    indep = np.sum(W**2, axis=1)                    # w_t' I w_t
    ll = -0.5 * T * logdet - 0.5 * np.sum(quad - indep)
    return ll


def t_copula_loglik(R, nu, U):
    """
    Log-likelihood of t-Copula with correlation R and df nu.
    Uses the formula from Demarta & McNeil (2005).
    """
    T, N  = U.shape
    # Convert uniform to t-quantiles
    t_inv = studentt.ppf(U, df=nu)   # shape (T, N)
    sign, logdet = np.linalg.slogdet(R)
    if sign <= 0:
        return -np.inf
    try:
        R_inv = np.linalg.inv(R)
    except np.linalg.LinAlgError:
        return -np.inf

    from scipy.special import gammaln
    ll = 0.0
    ll += T * (gammaln((nu + N) / 2) - gammaln(nu / 2) - (N/2) * np.log(np.pi * nu))
    ll -= 0.5 * T * logdet

    quad = np.einsum('ti,ij,tj->t', t_inv, R_inv, t_inv)   # (T,)
    indiv = np.sum(t_inv**2, axis=1)                         # sum of sq per row

    ll += np.sum(
        - ((nu + N) / 2) * np.log(1 + quad / nu)
        + np.sum(((nu + 1) / 2) * np.log(1 + t_inv**2 / nu), axis=1)
    )
    return ll


def fit_t_copula_nu(U, R_init, nu_range=(2.5, 30)):
    """Profile MLE over nu for t-copula (fix R = Kendall-tau based estimate)."""
    N = U.shape[1]

    def neg_ll(nu):
        nu = float(nu[0])
        return -t_copula_loglik(R_init, nu, U)

    best_ll, best_nu = -np.inf, 5.0
    for nu0 in [3.0, 5.0, 8.0, 12.0, 20.0]:
        try:
            res = minimize(neg_ll, [nu0], method='L-BFGS-B',
                           bounds=[(nu_range[0], nu_range[1])],
                           options={'maxiter': 200})
            if -res.fun > best_ll:
                best_ll  = -res.fun
                best_nu  = res.x[0]
        except Exception:
            continue
    return best_nu, best_ll


print('Fitting Gaussian copula...')
ll_gaussian = gaussian_copula_loglik(R_gaussian, W_matrix)
k_gaussian  = len(tickers_in) * (len(tickers_in) - 1) // 2  # free correlation params
aic_gaussian = -2 * ll_gaussian + 2 * k_gaussian
bic_gaussian = -2 * ll_gaussian + np.log(len(W_matrix)) * k_gaussian

print(f'Fitting t-Copula (profile MLE over df)...')
nu_fit, ll_t = fit_t_copula_nu(U_matrix, R_gaussian)
k_t  = k_gaussian + 1   # one extra parameter: nu
aic_t = -2 * ll_t + 2 * k_t
bic_t = -2 * ll_t + np.log(len(U_matrix)) * k_t

print(f'\n{"":20s}  {"Log-L":>12s}  {"k":>4s}  {"AIC":>12s}  {"BIC":>12s}')
print(f'{"Gaussian Copula":20s}  {ll_gaussian:12.1f}  {k_gaussian:4d}  {aic_gaussian:12.1f}  {bic_gaussian:12.1f}')
print(f'{"t-Copula":20s}  {ll_t:12.1f}  {k_t:4d}  {aic_t:12.1f}  {bic_t:12.1f}')

# Decision rule: lower AIC/BIC is preferred
aic_prefers_t = aic_t < aic_gaussian
bic_prefers_t = bic_t < bic_gaussian
tail_dep_prefers_t = avg_lambda > 0.05

votes_for_t = sum([aic_prefers_t, bic_prefers_t, tail_dep_prefers_t])
SELECTED_COPULA = 't' if votes_for_t >= 2 else 'gaussian'

print(f'\nCopula selection:')
print(f'  AIC prefers t-copula:           {aic_prefers_t}  (ΔAIC = {aic_gaussian - aic_t:.1f})')
print(f'  BIC prefers t-copula:           {bic_prefers_t}  (ΔBIC = {bic_gaussian - bic_t:.1f})')
print(f'  Tail dependence (λ_L > 0.05):  {tail_dep_prefers_t}  (avg λ_L = {avg_lambda:.3f})')
print(f'  → Selected copula: {SELECTED_COPULA.upper()}  ({votes_for_t}/3 votes for t-copula)')

if SELECTED_COPULA == 't':
    print(f'  → t-Copula df (nu) = {nu_fit:.2f}')

## 7. Delta-Gamma Approximation for USO and GLD

**Why not linear (delta-only) for commodities?**

USO (crude oil) and GLD (gold) exhibit:
- **Significant negative skewness** (oil in particular)
- **High excess kurtosis** — fat tails from supply shocks, geopolitical events
- **Non-linear P&L profile** if the ETF holds futures (roll risk, contango/backwardation)

**Delta-Gamma approximation:**  
$$\text{P\&L}_i \approx \delta_i \cdot r_i \cdot V_i + \frac{1}{2} \gamma_i \cdot r_i^2 \cdot V_i$$

For a long-only ETF: $\delta = 1$ (full price exposure). The gamma term captures second-order convexity effects.
In practice, gamma is calibrated from options or empirically estimated from P&L curvature.

**Note:** For USO specifically, the daily ETF return ≠ spot oil return due to futures roll.  
A full treatment would model the roll yield separately. Here we use the ETF price return directly  
but add the gamma correction to partially account for non-linearity.

In [ ]:
def portfolio_pnl_delta_gamma(sim_returns, weights_arr, portfolio_value, portfolio_config, tickers):
    """
    Compute simulated portfolio P&L with delta-gamma correction for tagged assets.

    sim_returns: (n_sims, n_assets) array of simulated log returns
    For linear assets: P&L_i = r_i * w_i * V
    For nonlinear (delta-gamma) assets:
      P&L_i = (delta * r_i + 0.5 * gamma * r_i^2) * w_i * V
    where delta = 1 for long ETF, gamma = gamma_proxy from config
    """
    n_sims, n_assets = sim_returns.shape
    pnl = np.zeros(n_sims)
    for j, t in enumerate(tickers):
        w_j  = weights_arr[j]
        r_j  = sim_returns[:, j]
        info = portfolio_config[t]
        if info.get('nonlinear', False):
            gamma = info.get('gamma_proxy', 0.02)
            # Delta-gamma: full first-order + half second-order
            pnl_j = (1.0 * r_j + 0.5 * gamma * r_j**2) * w_j * portfolio_value
        else:
            pnl_j = r_j * w_j * portfolio_value
        pnl += pnl_j
    return pnl


# Demonstrate the effect on a single asset
r_test = np.linspace(-0.10, 0.10, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, ticker in zip(axes, ['GLD', 'USO']):
    gamma = PORTFOLIO[ticker].get('gamma_proxy', 0.02)
    w     = PORTFOLIO[ticker]['weight']
    pnl_linear = r_test * w * PORTFOLIO_VALUE
    pnl_dg     = (r_test + 0.5 * gamma * r_test**2) * w * PORTFOLIO_VALUE
    ax.plot(r_test * 100, pnl_linear / 1e6, color='#4a90d9', lw=2, label='Delta (linear)')
    ax.plot(r_test * 100, pnl_dg / 1e6,     color='#e8734a', lw=2, label=f'Delta-Gamma (γ={gamma})')
    ax.fill_between(r_test*100, pnl_linear/1e6, pnl_dg/1e6, alpha=0.2, color='#9b72d0',
                    label='Gamma correction')
    ax.axhline(0, color='white', lw=0.5, linestyle=':')
    ax.axvline(0, color='white', lw=0.5, linestyle=':')
    ax.set_xlabel('Return (%)')
    ax.set_ylabel('P&L ($M)')
    ax.set_title(f'{ticker}: Linear vs Delta-Gamma P&L', pad=8)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Delta-Gamma Correction: Second-Order P&L for Commodity ETFs', y=1.02)
plt.tight_layout()
plt.savefig('output/delta_gamma.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Monte Carlo VaR — Copula-Based (DCC + t-Copula)

**Algorithm:**
1. Simulate $U \in [0,1]^N$ from selected copula (Gaussian or t) with DCC correlation $R_{T+1}$
2. Invert marginal CDFs: $r_i = F_i^{-1}(U_i)$ where $F_i$ is per-asset t-distribution
3. Compute portfolio P&L with delta-gamma correction for commodity assets
4. Extract VaR and ES at 99%

In [ ]:
N = len(tickers_in)

# ── Step 1: Simulate from copula using DCC forward-looking correlation ────────
print(f'Simulating {MC_SIMULATIONS:,} scenarios from {SELECTED_COPULA.upper()} copula...')

# Use the DCC forecast correlation for simulation (forward-looking)
R_sim = R_next.copy()
# Ensure numerical symmetry
R_sim = (R_sim + R_sim.T) / 2
np.fill_diagonal(R_sim, 1.0)

# Cholesky decomposition of correlation matrix
try:
    L_chol = np.linalg.cholesky(R_sim)
except np.linalg.LinAlgError:
    print('  R_sim not PD, applying small regularisation...')
    R_sim += 1e-6 * np.eye(N)
    L_chol = np.linalg.cholesky(R_sim)

if SELECTED_COPULA == 't':
    # t-Copula simulation
    Z_sim = RNG.standard_normal((MC_SIMULATIONS, N))  # standard normal
    chi2_draws = RNG.chisquare(df=nu_fit, size=MC_SIMULATIONS)
    # Multivariate t: X = Z / sqrt(chi2/nu), then correlate
    W_copula = Z_sim @ L_chol.T / np.sqrt(chi2_draws / nu_fit)[:, None]
    # Convert to uniform via t CDF
    U_sim = studentt.cdf(W_copula, df=nu_fit)
else:
    # Gaussian copula simulation
    Z_sim    = RNG.standard_normal((MC_SIMULATIONS, N))
    W_copula = Z_sim @ L_chol.T
    U_sim    = norm.cdf(W_copula)

U_sim = np.clip(U_sim, 1e-6, 1 - 1e-6)

# ── Step 2: Invert per-asset marginals ────────────────────────────────────────
# Use GARCH conditional vol to rescale: r_it = z_it * sigma_{T+1,i}
Z_rescaled = np.empty_like(U_sim)
for j, ticker in enumerate(tickers_in):
    df_j, loc_j, scale_j = marginal_params[ticker]
    sigma_next_j = garch_sigmas_next[j]  # GARCH 1-step forecast
    # Invert the per-asset t-distribution to get GARCH residual z
    z_j = studentt.ppf(U_sim[:, j], df=df_j, loc=loc_j, scale=scale_j)
    # Rescale by forward-looking GARCH vol
    Z_rescaled[:, j] = z_j * sigma_next_j

# ── Step 3: Compute portfolio P&L with delta-gamma correction ─────────────────
pnl_copula = portfolio_pnl_delta_gamma(
    Z_rescaled, w_arr, PORTFOLIO_VALUE, PORTFOLIO, tickers_in
)

# ── Step 4: VaR and ES ────────────────────────────────────────────────────────
threshold_copula = np.percentile(pnl_copula, (1 - VAR_CONFIDENCE) * 100)
var_mc_copula    = -threshold_copula
tail_pnl_copula  = pnl_copula[pnl_copula <= threshold_copula]
es_mc_copula     = -tail_pnl_copula.mean()

print(f'\nMC VaR ({SELECTED_COPULA.upper()} copula + DCC + delta-gamma):')
print(f'  VaR (99%, 1-day): ${var_mc_copula:,.0f}  ({var_mc_copula/PORTFOLIO_VALUE*100:.2f}%)')
print(f'  ES  (99%, 1-day): ${es_mc_copula:,.0f}   ({es_mc_copula/PORTFOLIO_VALUE*100:.2f}%)')
print(f'  ES/VaR ratio:     {es_mc_copula/var_mc_copula:.3f}')

# ── Comparison: what would linear (no gamma) give? ────────────────────────────
pnl_linear_only  = Z_rescaled @ w_arr * PORTFOLIO_VALUE
var_mc_linear    = -np.percentile(pnl_linear_only, (1 - VAR_CONFIDENCE) * 100)
print(f'\n  [Linear approx VaR]:  ${var_mc_linear:,.0f}')
print(f'  Delta-gamma premium:  ${var_mc_copula - var_mc_linear:,.0f}  '
      f'({(var_mc_copula/var_mc_linear - 1)*100:.2f}%)')

In [ ]:
# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bins = np.linspace(np.percentile(pnl_copula, 0.1), np.percentile(pnl_copula, 99.9), 150)

axes[0].hist(pnl_copula/1e6, bins=bins/1e6, alpha=0.6, color='#4a90d9',
             density=True, label=f'{SELECTED_COPULA.upper()} Copula')
axes[0].hist(pnl_linear_only/1e6, bins=bins/1e6, alpha=0.4, color='#e8734a',
             density=True, label='Linear (no gamma)')
axes[0].axvline(-var_mc_copula/1e6, color='#4a90d9', lw=2, linestyle='--',
                label=f'VaR: ${var_mc_copula/1e6:.2f}M')
axes[0].set_xlabel('Daily P&L ($M)')
axes[0].set_title('Full Distribution')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Tail zoom
tail_cut = np.percentile(pnl_copula, 2)
tail_bins = np.linspace(tail_cut, np.percentile(pnl_copula, 8), 80)
axes[1].hist(pnl_copula/1e6, bins=tail_bins/1e6, alpha=0.6, color='#4a90d9',
             density=True, label=f'{SELECTED_COPULA.upper()} Copula tail')
axes[1].hist(pnl_linear_only/1e6, bins=tail_bins/1e6, alpha=0.4, color='#e8734a',
             density=True, label='Linear tail')
axes[1].axvline(-var_mc_copula/1e6, color='#4a90d9', lw=2, linestyle='--')
axes[1].set_xlabel('Daily P&L ($M)')
axes[1].set_title('Left Tail Zoom')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.suptitle(f'Copula MC VaR ({MC_SIMULATIONS:,} sims, {SELECTED_COPULA.upper()}, DCC + Delta-Gamma)', y=1.01)
plt.tight_layout()
plt.savefig('output/mc_copula_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Historical Simulation VaR + ES (Rolling 250 trading days)

Rolling window uses **exactly 250 trading days** (not calendar days).

In [ ]:
# Portfolio P&L with delta-gamma for historical returns
hist_returns_arr = returns[tickers_in].values   # (T, N)
pnl_hist_dg = portfolio_pnl_delta_gamma(
    hist_returns_arr, w_arr, PORTFOLIO_VALUE, PORTFOLIO, tickers_in
)
pnl_series = pd.Series(pnl_hist_dg, index=returns.index)

def rolling_var_es(pnl_arr, confidence, window):
    """Exactly 250 trading-day rolling VaR and ES."""
    n   = len(pnl_arr)
    var_list, es_list = [], []
    for i in range(window, n + 1):
        w_pnl = pnl_arr[i - window: i]
        thresh = np.percentile(w_pnl, (1 - confidence) * 100)
        var_list.append(-thresh)
        tail = w_pnl[w_pnl <= thresh]
        es_list.append(-tail.mean() if len(tail) > 0 else -thresh)
    idx = pnl_series.index[window - 1: n]
    return (pd.Series(var_list, index=idx, name='HS_VaR'),
            pd.Series(es_list,  index=idx, name='HS_ES'))

hs_var, hs_es = rolling_var_es(pnl_series.values, VAR_CONFIDENCE, HS_WINDOW)
hs_var_latest = hs_var.iloc[-1]

print(f'HS VaR (99%, 1-day, exactly 250 trading days): ${hs_var_latest:,.0f}  '
      f'({hs_var_latest/PORTFOLIO_VALUE*100:.2f}%)')
print(f'HS ES  (99%, 1-day):                          ${hs_es.iloc[-1]:,.0f}  '
      f'({hs_es.iloc[-1]/PORTFOLIO_VALUE*100:.2f}%)')
print(f'ES/VaR ratio: {hs_es.iloc[-1]/hs_var_latest:.3f}')

## 10. Stressed VaR — Exactly 250 Trading Days

**FIX from v1:** v1 used calendar-year dates (`2008-01-01` to `2008-12-31`), which can yield ~252 or ~250 days depending on the year, and does not guarantee exactly 250 trading days as required by Basel 2.5.

Here we:
1. Scan all rolling 250-trading-day windows
2. Find the one that maximises VaR for the **current portfolio weights**
3. Also report the 2008 anchored window as a named scenario

In [ ]:
def scan_stressed_var(pnl_series, window=250, confidence=0.99):
    """
    Scan all rolling 250-trading-day windows to find the worst stressed VaR.
    Returns the VaR, start date, end date, and window index.
    Uses exactly `window` trading days (not calendar days).
    """
    arr   = pnl_series.values
    dates = pnl_series.index
    n     = len(arr)
    best_svar, best_i = 0.0, 0

    for i in range(n - window + 1):
        w = arr[i: i + window]
        svar_i = -np.percentile(w, (1 - confidence) * 100)
        if svar_i > best_svar:
            best_svar, best_i = svar_i, i

    start = dates[best_i]
    end   = dates[best_i + window - 1]
    return best_svar, start, end


def stressed_var_anchored(pnl_series, anchor_start_str, window=250, confidence=0.99):
    """
    Compute stressed VaR starting from a given date,
    using exactly `window` trading days forward.
    """
    arr   = pnl_series.values
    dates = pnl_series.index
    # Find nearest date at or after anchor_start
    anchor_ts = pd.Timestamp(anchor_start_str)
    idx_start = dates.searchsorted(anchor_ts)
    idx_start = min(idx_start, len(dates) - window)
    w = arr[idx_start: idx_start + window]
    svar = -np.percentile(w, (1 - confidence) * 100)
    return svar, dates[idx_start], dates[idx_start + window - 1]


print('Scanning all 250-trading-day windows for worst stressed VaR...')
svar_worst, svar_start, svar_end = scan_stressed_var(pnl_series, window=HS_WINDOW)
print(f'  Worst window: {svar_start.date()} → {svar_end.date()}  ({HS_WINDOW} trading days exactly)')
print(f'  SVaR (worst, 99%): ${svar_worst:,.0f}  ({svar_worst/PORTFOLIO_VALUE*100:.2f}%)')

# Named historical scenarios — anchored to crisis onset
CRISIS_ANCHORS = {
    'GFC 2008 (Lehman)':   '2008-09-01',
    'Euro Crisis 2010':    '2010-05-01',
    'COVID 2020':          '2020-02-01',
}
print('\nNamed 250-trading-day stress windows (crisis anchored):')
named_svars = {}
for name, anchor in CRISIS_ANCHORS.items():
    sv, s, e = stressed_var_anchored(pnl_series, anchor)
    named_svars[name] = sv
    print(f'  {name:30s}: {s.date()} → {e.date()}  |  SVaR=${sv:,.0f}  ({sv/PORTFOLIO_VALUE*100:.2f}%)')

# Use the worst window as the regulatory SVaR
svar_regulatory = svar_worst
print(f'\nRegulatory SVaR (Basel 2.5, worst 250-day window): ${svar_regulatory:,.0f}')

## 11. Liquidity-Adjusted VaR (LVaR) — Dynamic Spreads

In [ ]:
rows = []
total_liq = 0.0
latest_date_lvar = pnl_series.index[-1]

for ticker, info in PORTFOLIO.items():
    if ticker not in tickers_in:
        continue
    ac       = info['asset_class']
    w        = info['weight']
    notional = w * PORTFOLIO_VALUE
    ldays    = LIQUIDATION_DAYS.get(ac, 1)

    spread_static  = BID_ASK_SPREADS_STATIC.get(ac, 0.01)
    spread_dynamic = get_dynamic_spread(ac, latest_date_lvar)

    # LVaR formula: 0.5 * spread * notional * sqrt(T)
    # Use dynamic spread if available, else static
    lcost = 0.5 * spread_dynamic * notional * np.sqrt(ldays)
    total_liq += lcost

    rows.append({
        'Ticker':          ticker,
        'Asset Class':     ac,
        'Weight':          f'{w*100:.1f}%',
        'Notional ($M)':   f'{notional/1e6:.1f}',
        'Spread Static':   f'{spread_static*100:.3f}%',
        'Spread Dynamic':  f'{spread_dynamic*100:.3f}%',
        'Liq Days':        ldays,
        'Liq Cost ($K)':   f'{lcost/1e3:.2f}',
    })

lvar = hs_var_latest + total_liq
lvar_df = pd.DataFrame(rows)
lvar_df.to_csv('output/lvar_breakdown.csv', index=False)

print(f'Base VaR (HS, delta-gamma): ${hs_var_latest:,.0f}')
print(f'Total liquidity cost:       ${total_liq:,.0f}')
print(f'LVaR:                       ${lvar:,.0f}  ({lvar/PORTFOLIO_VALUE*100:.2f}%)')
print(f'Liquidity premium:          {total_liq/hs_var_latest*100:.1f}% of base VaR')
print()
print(lvar_df.to_string(index=False))

## 12. Backtesting — Basel Traffic Light

**FIX from v1:** Christoffersen test now uses numpy vectorized operations (was O(n) Python generator).

In [ ]:
def kupiec_pof(exceptions, n_obs, confidence):
    """Kupiec (1995) POF test. H0: coverage is correct at level (1-confidence)."""
    p = 1 - confidence
    if exceptions == 0:
        return {'LR': 0.0, 'p_value': 1.0, 'reject_H0': False}
    p_hat = exceptions / n_obs
    # Guard against degenerate case
    if p_hat <= 0 or p_hat >= 1:
        return {'LR': np.inf, 'p_value': 0.0, 'reject_H0': True}
    lr = -2 * (
        np.log((1-p)**(n_obs-exceptions) * p**exceptions)
        - np.log((1-p_hat)**(n_obs-exceptions) * p_hat**exceptions)
    )
    p_val = 1 - chi2.cdf(lr, df=1)
    return {'LR': lr, 'p_value': p_val, 'reject_H0': p_val < 0.05}


def christoffersen_ind(exc_series):
    """
    Christoffersen (1998) independence test — vectorized (FIX from v1).
    H0: exceptions are serially independent.
    """
    hits = exc_series.astype(int).values
    # Vectorized transition counts
    h_t   = hits[:-1]
    h_t1  = hits[1:]
    n00 = np.sum((h_t == 0) & (h_t1 == 0))
    n01 = np.sum((h_t == 0) & (h_t1 == 1))
    n10 = np.sum((h_t == 1) & (h_t1 == 0))
    n11 = np.sum((h_t == 1) & (h_t1 == 1))

    denom_01 = n00 + n01
    denom_11 = n10 + n11
    if denom_01 == 0 or denom_11 == 0:
        return {'LR_ind': 0.0, 'p_value': 1.0, 'clustering': False}

    pi_01 = n01 / denom_01
    pi_11 = n11 / denom_11
    pi    = (n01 + n11) / (n00 + n01 + n10 + n11)

    def safe_log(x):
        return np.log(x) if x > 1e-10 else -1e10

    lr_ind = -2 * (
        (n00 + n10) * safe_log(1 - pi) + (n01 + n11) * safe_log(pi)
        - n00 * safe_log(1 - pi_01) - n01 * safe_log(pi_01 if pi_01 > 1e-10 else 1e-10)
        - n10 * safe_log(1 - pi_11) - n11 * safe_log(pi_11 if pi_11 > 1e-10 else 1e-10)
    )
    p_val = 1 - chi2.cdf(lr_ind, df=1)
    return {'LR_ind': lr_ind, 'p_value': p_val, 'clustering': p_val < 0.05}


# ── Run backtest ──────────────────────────────────────────────────────────────
aligned = pd.DataFrame({
    'pnl': pnl_series,
    'var': hs_var,
}).dropna()
aligned['exception'] = aligned['pnl'] < -aligned['var']

n_exc = aligned['exception'].sum()
n_obs = len(aligned)
exp_exc = int((1 - VAR_CONFIDENCE) * n_obs)
zone = 'GREEN' if n_exc <= 4 else ('YELLOW' if n_exc <= 9 else 'RED')
emoji = {'GREEN': '🟢', 'YELLOW': '🟡', 'RED': '🔴'}[zone]

kup = kupiec_pof(n_exc, n_obs, VAR_CONFIDENCE)
ch  = christoffersen_ind(aligned['exception'])

print(f'Observations: {n_obs}  |  Exceptions: {n_exc}  (expected ≈ {exp_exc})')
print(f'Basel Traffic Light: {emoji} {zone}')
print(f'Kupiec POF   — LR={kup["LR"]:.3f}, p={kup["p_value"]:.3f}, reject H0={kup["reject_H0"]}')
print(f'Christoffersen — LR={ch["LR_ind"]:.3f}, p={ch["p_value"]:.3f}, clustering={ch["clustering"]}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(aligned.index, aligned['pnl']/1e6, alpha=0.12, color='#4ac9a0')
ax.plot(aligned.index, aligned['pnl']/1e6, color='#4ac9a0', lw=0.7, label='Daily P&L (delta-gamma)')
ax.plot(aligned.index, -aligned['var']/1e6, color='#4a90d9', lw=1.2,
        linestyle='--', label='-VaR threshold')
exc = aligned[aligned['exception']]
ax.scatter(exc.index, exc['pnl']/1e6, color='#e05252', s=20, zorder=5,
           label=f'Exceptions (n={n_exc})')
ax.legend(fontsize=9)
ax.set_ylabel('P&L ($M)')
ax.set_title(f'Backtesting: P&L vs VaR  |  {emoji} {zone}  |  Exceptions: {n_exc}', pad=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/backtest_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Expected Shortfall + McNeil-Frey Backtest

In [ ]:
def mcneil_frey_test(pnl, var_series, es_series):
    """
    McNeil & Frey (2000): Z_t = (PnL_t + VaR_t) / ES_t on exception days.
    H0: E[Z|exception] = 0  (ES correctly specified)
    Z < 0 → ES understates tail risk.
    """
    df = pd.DataFrame({'pnl': pnl, 'var': var_series, 'es': es_series}).dropna()
    df['exception'] = df['pnl'] < -df['var']
    exc = df[df['exception']].copy()
    if len(exc) < 5:
        print(f'Only {len(exc)} exceptions — too few for reliable test')
        return None
    exc['Z'] = (exc['pnl'] + exc['var']) / exc['es']
    t_stat, p_val = stats.ttest_1samp(exc['Z'].dropna(), popmean=0)
    reject    = p_val < 0.05
    direction = 'understates' if t_stat < 0 else 'overstates'
    print(f'McNeil-Frey ES backtest:')
    print(f'  Exception days: {len(exc)}')
    print(f'  Mean Z:         {exc["Z"].mean():.4f}  (0 = perfect)')
    print(f'  t-stat / p:     {t_stat:.3f} / {p_val:.4f}')
    if reject:
        print(f'  → Reject H0: ES {direction} true tail loss')
    else:
        print(f'  → Cannot reject H0: ES appears well-calibrated')
    return exc


mcneil_frey_test(pnl_series, hs_var, hs_es)

## 14. GARCH-Filtered HS VaR (Ghost Effect Fix)

In [ ]:
# Use portfolio GARCH (fit on portfolio-level returns for GARCH-HS)
port_ret_arr = pnl_series.values / PORTFOLIO_VALUE
g_port = GARCH11().fit(port_ret_arr)

print(f'Portfolio GARCH(1,1):')
print(f'  omega={g_port.omega_:.6f}  alpha={g_port.alpha_:.4f}  beta={g_port.beta_:.4f}')
print(f'  persistence={g_port.persistence:.4f}  LR Ann Vol={g_port.long_run_ann_vol:.1f}%')

def garch_filtered_var_es(pnl, garch_model, confidence, window, pv):
    r  = (pnl / pv).values
    z  = garch_model.resid_
    s2 = garch_model.sigma2_
    om, al, be = garch_model.omega_, garch_model.alpha_, garch_model.beta_

    var_list, es_list, idx = [], [], []
    for i in range(window, len(r)):
        z_win    = z[i - window: i]
        sig_next = np.sqrt(om + al * r[i-1]**2 + be * s2[i-1])
        sim_pnl  = z_win * sig_next * pv
        thresh   = np.percentile(sim_pnl, (1 - confidence) * 100)
        var_list.append(-thresh)
        tail = sim_pnl[sim_pnl <= thresh]
        es_list.append(-tail.mean() if len(tail) > 0 else -thresh)
        idx.append(pnl.index[i])

    return (pd.Series(var_list, index=idx, name='GARCH_VaR'),
            pd.Series(es_list,  index=idx, name='GARCH_ES'))

garch_var, garch_es = garch_filtered_var_es(pnl_series, g_port, VAR_CONFIDENCE, HS_WINDOW, PORTFOLIO_VALUE)
print(f'\nGARCH-Filtered VaR (99%): ${garch_var.iloc[-1]:,.0f}')
print(f'GARCH-Filtered ES  (99%): ${garch_es.iloc[-1]:,.0f}')

## 15. Complete VaR Summary

In [ ]:
metrics = [
    ('HS VaR (delta-gamma)',              hs_var_latest,         None),
    ('HS ES (delta-gamma)',               hs_es.iloc[-1],        None),
    ('Stressed VaR (worst 250-day)',      svar_regulatory,       None),
    (f'MC VaR ({SELECTED_COPULA.upper()} Copula+DCC)', var_mc_copula, None),
    (f'MC ES  ({SELECTED_COPULA.upper()} Copula+DCC)', es_mc_copula,  None),
    ('GARCH-Filtered VaR',               garch_var.iloc[-1],    None),
    ('GARCH-Filtered ES',                garch_es.iloc[-1],     None),
    ('LVaR (dynamic spread)',            lvar,                   None),
]
for name, anchor in CRISIS_ANCHORS.items():
    sv = named_svars.get(name, 0)
    metrics.append((f'SVaR: {name}', sv, None))

summary_df = pd.DataFrame([
    {'Metric': m, 'VaR/ES ($)': v, 'VaR/ES ($M)': v/1e6, '% of Portfolio': v/PORTFOLIO_VALUE*100,
     'vs HS VaR': v/hs_var_latest}
    for m, v, _ in metrics
]).round({'VaR/ES ($M)': 3, '% of Portfolio': 3, 'vs HS VaR': 3})

print('=== VaR Engine v3 — Full Summary ===')
print(summary_df.to_string(index=False))
summary_df.to_csv('output/var_summary_v3.csv', index=False)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#4a90d9','#5ba3e8','#e8734a','#4ac9a0','#5fd9b0',
          '#9b72d0','#ae88e0','#e05252','#f5c842','#f7d35e','#a0c840']
vals  = summary_df['VaR/ES ($M)'].values
names = summary_df['Metric'].values
cols  = colors[:len(vals)]

bars = ax.barh(names, vals, color=cols, alpha=0.85, height=0.65)
for bar, val in zip(bars, vals):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'${val:.3f}M', va='center', color='#e8e8e8', fontsize=9)
ax.set_xlabel('Risk Measure ($M)')
ax.set_title('VaR Engine v3 — All Risk Metrics', pad=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/var_summary_v3.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== All outputs saved to ./output/ ===')
for f in sorted(os.listdir('output')):
    path = os.path.join('output', f)
    print(f'  {f:50s}  {os.path.getsize(path)/1024:.1f} KB')

## 16. Key Improvements Summary (v1 → v3)

| Issue in v1/v2 | Fix in v3 |
|---|---|
| Student-t MC: fitted df per asset, took average, used single df | **Copula framework**: per-asset t marginals + Gaussian/t-Copula for joint structure |
| No formal copula selection | **AIC/BIC + tail dependence test** — 3-vote selection rule |
| Static sample covariance, no PD check | **Ledoit-Wolf shrinkage** if eigenvalue < threshold or condition # > 1e6 |
| Stress window = calendar year (not exactly 250 TD) | **Scan all 250-trading-day windows**, also named crisis anchors |
| Single portfolio-level GARCH | **Per-asset GARCH(1,1) + DCC** for time-varying correlation |
| Static bid-ask spread | **VIX-scaled dynamic spread** with asset-class sensitivity; graceful fallback |
| Linear P&L for all assets | **Delta-gamma approximation** for USO and GLD (commodity convexity) |
| Christoffersen: slow Python generator | **Numpy vectorized** transition count |
| GARCH warm-up: full-sample variance | **Rolling 30-day warm-up** for better initial condition |
| Shapiro-Wilk on residuals (wrong diagnostic) | **Ljung-Box on z²** — correct test for remaining ARCH effects |
| Scattered RNG with single seed | **Global RNG object** passed consistently |

### Remaining Limitations (production upgrade path)

1. **Copula calibration**: A full IFM (Inference Functions for Margins) approach would jointly optimise marginals and copula parameters — here we use a two-step approach for tractability.
2. **Roll yield for USO**: The ETF tracks crude oil futures, not spot. A complete model would decompose spot return + roll yield + tracking error.
3. **GARCH-t innovations**: GARCH with Normal innovations leaves fat tails in residuals. GARCH-t (Student innovations) or EGARCH-skewed-t would handle this better.
4. **DCC scalability**: Engle's DCC is O(N²) per time step. For large N, use factor DCC or hierarchical correlation models.
5. **Bid-ask spreads**: VIX is a rough proxy. Production would use TRACE tick data for credit, TAQ for equity, to construct realised spread series.
6. **Intraday risk**: 1-day VaR misses intraday gap risk. High-frequency extensions would use realised covariance matrices.
7. **Regime-switching**: A Markov-switching GARCH would better capture structural breaks (crisis vs calm regimes).